# Feature Engineering — IEEE-CIS Fraud Detection

01_eda.ipynb'deki bulgulara dayanarak proje kapsamındaki türetilmiş özellikler kodlanır. Gerçek fonksiyonlar `ml/src/features.py`'de tanımlanır (eğitim ve ileride FastAPI ML servisi tarafından ortak kullanılacaktır); bu notebook onları import edip doğrular.

Tüm özellikler **leakage-safe**'tir: bir işlem için hesaplanan değerler yalnızca o işlemden ÖNCEki geçmişi kullanır, gelecekteki ya da işlemin kendi bilgisini sızdırmaz.

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from features import (
    add_uid,
    add_avg_transaction_amount,
    add_amount_deviation_from_user,
    add_transaction_counts,
)

pd.set_option("display.max_columns", 50)

## Veri Yükleme

In [ ]:
df_transaction = pd.read_csv("../data/train_transaction.csv")
df_identity = pd.read_csv("../data/train_identity.csv")
df = pd.merge(df_transaction, df_identity, on="TransactionID", how="left")

## 1. Pseudo-Kullanıcı Kimliği (`uid`)

Veri setinde açık bir `user_id` yok. `card1 + card2 + card3 + card5 + addr1 + D1n` kombinasyonu, aynı kullanıcıyı/kartı güçlü ihtimalle işaret eden bir pseudo-kimlik olarak kullanılır (Kaggle topluluğunda yaygın kabul gören bir yaklaşım). `D1n`, zamanla artan `D1` sütununun (kartın ilk işleminden bu yana geçen gün) işlem gününe göre normalize edilmiş hali — bu sayede aynı kullanıcının farklı zamanlardaki işlemleri yanlışlıkla farklı gruplara bölünmüyor. Bu kesin bir kullanıcı ID'si değildir — bir yaklaşıklıktır.

In [ ]:
df = add_uid(df)
df["uid"].nunique()

**Bulgu:** `D1` normalizasyonu öncesi 232,821 olan benzersiz `uid` sayısı, normalizasyon sonrası **197,807**'ye düştü (ortalama kullanıcı başına ~3 işlem) — daha tutarlı bir gruplama.

## 2. Ortalama İşlem Tutarı (`avg_transaction_amount`)

Her `uid` için, o ana kadarki geçmiş işlemlerin ortalama tutarı. Basit bir `groupby().mean()` veri sızıntısına yol açar (kullanıcının tüm işlemlerini, geçmiş+gelecek+kendisi dahil, kullanır); bunun yerine `shift(1)` (mevcut işlemi hariç tut) + `expanding().mean()` (o ana kadar birikmiş ortalama) kombinasyonu kullanılır.

In [ ]:
df = add_avg_transaction_amount(df)
df[["uid", "TransactionDT", "TransactionAmt", "avg_transaction_amount"]].head(10)

**Bulgu:** Bir kullanıcının ilk işleminde henüz geçmiş olmadığı için sonuç `NaN` (beklenen). Tekrarlanan `uid`'lerde elle doğrulandı: ortalama, işlemin kendi tutarını değil, yalnızca önceki işlem(ler)i yansıtıyor — sızıntı yok. `NaN` değerlerinin nasıl ele alınacağı (olduğu gibi bırakma / doldurma / bayrak ekleme) modelleme aşamasında, seçilen modele göre netleştirilecek.

## 3. Kullanıcı Ortalamasından Sapma (`amount_deviation_from_user`)

`(TransactionAmt - avg_transaction_amount) / avg_transaction_amount` — mutlak fark yerine yüzdesel sapma kullanılır, böylece farklı harcama seviyelerindeki kullanıcılar karşılaştırılabilir olur.

In [ ]:
df = add_amount_deviation_from_user(df)
df[["uid", "TransactionAmt", "avg_transaction_amount", "amount_deviation_from_user"]].head(10)

**Bulgu:** Negatif değer kullanıcının kendi ortalamasının altında, pozitif değer üstünde bir harcamayı gösteriyor (örn. `-0.33` = ortalamadan %33 daha düşük). Elle doğrulama ile hesaplamanın doğru çalıştığı teyit edildi.

## 4. Zaman Penceresi İşlem Sayıları (`transactions_last_10min`, `transactions_last_24h`)

Her işlem için, aynı `uid`'in kendisinden önceki 10 dakika ve 24 saat içindeki işlem sayısı — zaman tabanlı `rolling()` penceresi (`closed="left"` ile mevcut işlem hariç tutulur).

İki teknik detay gerekli oldu:
- **`min_periods=0`**: pencerede hiç işlem yoksa sonucun `NaN` değil `0` olması için (0 işlem, geçerli bir cevaptır).
- **Nanosaniyelik tie-breaker**: aynı `uid`'in aynı saniyeye denk gelen birden fazla işlemi olursa, zaman tabanlı `rolling()` bunları ayırt edemeyip satır kaybedebiliyor; her satıra grup-içi sırasına göre çok küçük bir zaman ofseti eklenerek bu önlendi (10dk/24s pencere sınırlarını etkilemeyecek kadar küçük).

In [ ]:
df = add_transaction_counts(df)
df[["uid", "TransactionDT", "transactions_last_10min", "transactions_last_24h"]].head(20)

**Bulgu:** Sonuçlar elle doğrulandı — örneğin bir kullanıcının iki işlemi arasında 333 saniye (~5.5 dakika) varsa `transactions_last_10min=1`, 7190 saniye (~2 saat) varsa `transactions_last_10min=0` ama `transactions_last_24h` doğru şekilde önceki işlemleri sayıyor.

## Durum

Tamamlanan özellikler: `uid` (pseudo-kullanıcı kimliği), `avg_transaction_amount`, `amount_deviation_from_user`, `transactions_last_10min`, `transactions_last_24h`.

Kalan özellikler (proje kapsamındaki listeye göre): `new_device`, `new_location`, `distance_from_last_transaction`, `merchant_risk`. `failed_attempts_last_hour` veri setinde mevcut olmadığı için kapsam dışı bırakıldı.